# Credit Card Fraud Detection

**Goal:** Detect fraudulent transactions (imbalanced data)
**Algorithm:** Random Forest + Undersampling
**Dataset:** [Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
else:
    print('Running locally')

Running locally - ready


## 1. Load Data

In [1]:
path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
df = pd.read_csv(f'{path}/creditcard.csv')
print('Shape:', df.shape)
print('Fraud: %d / %d (%.4f%%)' % (df['Class'].sum(), len(df), df['Class'].mean()*100))

Shape: (284807, 31)
Fraud: 492 / 284807 (0.1727%)


## 2. Handle Imbalance

In [1]:
X, y = df.drop('Class', axis=1), df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
fraud = X_train[y_train==1]
normal = X_train[y_train==0].sample(n=len(fraud), random_state=42)
Xb = pd.concat([fraud, normal])
yb = np.array([1]*len(fraud) + [0]*len(fraud))
print('Balanced:', len(Xb), 'samples')

Balanced: 688 samples


## 3. Train & Evaluate

In [1]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(Xb, yb)
y_pred = model.predict(X_test)
print('ROC-AUC: %.4f' % roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
print(classification_report(y_test, y_pred, target_names=['Normal','Fraud']))

ROC-AUC: 0.9723
              precision    recall  f1-score   support
      Normal       1.00      1.00      1.00     85315
       Fraud       0.76      0.91      0.83       128
    accuracy                           1.00     85443
